[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ADRIANVM117/quant-research-notes/blob/factor-momentum-colab-v1/Momentum_systematic_strategies/02_factor_momentum/strategy_candidates/xle_systematic_momentum/notebooks/01_execution_feasibility.ipynb)

> El enlace público funcionará cuando esta versión esté publicada en GitHub. En Colab, ejecuta todas las celdas: el bootstrap descarga y verifica los siete insumos Gold y la notebook de referencia fijada, sin requerir claves ni ejecuciones previas.

In [ ]:
# Bootstrap reproducible para Google Colab. No modifica la ejecución local.
from datetime import date
from hashlib import sha256
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import urlopen
import json
import os
import subprocess
import sys

try:
    import google.colab
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    # Colab puede cargar NumPy antes de esta celda. Reinstalarlo no sustituye el módulo en memoria.
    SUPPORTED_NUMPY_VERSIONS = ("2.4.6", "2.5.4")
    loaded_numpy = sys.modules.get("numpy")
    try:
        numpy_distribution = version("numpy")
    except PackageNotFoundError:
        numpy_distribution = None
    numpy_runtime = getattr(loaded_numpy, "__version__", numpy_distribution)
    if numpy_runtime not in SUPPORTED_NUMPY_VERSIONS:
        raise RuntimeError(f"NumPy {numpy_runtime!r} no pertenece al contrato reproducible {SUPPORTED_NUMPY_VERSIONS}. Reinicia el entorno Colab limpio.")
    REQUIRED_PACKAGES = {"polars": "1.44.2", "plotly": "6.2.0"}
    installed = {}
    for package, required in REQUIRED_PACKAGES.items():
        loaded = sys.modules.get(package)
        if loaded is not None and getattr(loaded, "__version__", None) != required:
            raise RuntimeError(f"{package} ya está cargado con una versión incompatible. Reinicia el entorno Colab limpio.")
        try:
            installed[package] = version(package)
        except PackageNotFoundError:
            installed[package] = None
    if any(installed[name] != required for name, required in REQUIRED_PACKAGES.items()):
        command = [sys.executable, "-m", "pip", "install", "--quiet"]
        command.extend(f"{package}=={required}" for package, required in REQUIRED_PACKAGES.items())
        subprocess.check_call(command)

    import numpy as np
    import plotly
    import polars as pl

    if np.__version__ not in SUPPORTED_NUMPY_VERSIONS or pl.__version__ != "1.44.2" or plotly.__version__ != "6.2.0":
        raise RuntimeError("Las versiones instaladas no coinciden con el entorno reproducible fijado.")

    SOURCE_COMMIT = "a8ab4d6bf04184551a783925eec1a94db0a31dd9"
    RAW_ROOT = ("https://raw.githubusercontent.com/ADRIANVM117/quant-research-notes/"
                f"{SOURCE_COMMIT}/Momentum_systematic_strategies/data/gold/daily_series/etf")
    INPUTS = {
        "SPY": "b0767b45c2d67dbe8d99e11e1603682fd28468c31de65d037a4c0a50fefa67e9",
        "QQQ": "69760c426b11238aca1f1200f1c44933938954115d944db3b5a3df77618475fc",
        "IWM": "7913707e8a33b60780a7fe93617eb8114ef71b0e66e6a3b0b4dbab752cf09523",
        "IWD": "d73576d8c83dcbe57392174cb341061321ab963cd4adbdd063f455342caea0e0",
        "IWF": "d0c4165277bc9167421dde15ef9ede395bc35d259326de22913a76ef5792314e",
        "XLE": "f98e7468b5aee758298229a14443818ea0148bc46b26042a02abe1dee9af7a5a",
        "XLF": "52612673ab44bb3e222d5ac208bd3b9dfd0871c2259ff9fb472c59d902f170e0",
    }
    PROJECT_ROOT = Path("/content/quant-research-notes/Momentum_systematic_strategies")
    GOLD_ROOT = PROJECT_ROOT / "data" / "gold" / "daily_series" / "etf"
    GOLD_ROOT.mkdir(parents=True, exist_ok=True)

    for symbol, expected_hash in INPUTS.items():
        destination = GOLD_ROOT / f"{symbol}.parquet"
        if destination.exists():
            observed_hash = sha256(destination.read_bytes()).hexdigest()
            if observed_hash != expected_hash:
                raise RuntimeError(f"{symbol}: existe un archivo local con SHA-256 distinto; no se reemplaza.")
        else:
            url = f"{RAW_ROOT}/{symbol}.parquet"
            try:
                with urlopen(url, timeout=60) as response:
                    if response.status != 200:
                        raise RuntimeError(f"{symbol}: HTTP {response.status} al descargar {url}")
                    payload = response.read()
            except (HTTPError, URLError, TimeoutError) as error:
                raise RuntimeError(f"{symbol}: no fue posible descargar el insumo fijado.") from error
            observed_hash = sha256(payload).hexdigest()
            if observed_hash != expected_hash:
                raise RuntimeError(f"{symbol}: SHA-256 remoto inesperado; se cancela la ejecución.")
            temporary = destination.with_suffix(".parquet.download")
            temporary.write_bytes(payload)
            temporary.replace(destination)

    required_columns = {"session_date", "adjusted_close"}
    expected_start, expected_end, expected_sessions = date(2018, 1, 2), date(2026, 11, 17), 2232
    calendars, COLAB_INPUT_AUDIT = [], []
    for symbol, expected_hash in INPUTS.items():
        path = GOLD_ROOT / f"{symbol}.parquet"
        frame = pl.read_parquet(path).with_columns(pl.col("session_date").cast(pl.Date))
        missing = required_columns.difference(frame.columns)
        if missing:
            raise RuntimeError(f"{symbol}: faltan columnas requeridas: {sorted(missing)}")
        scoped = frame.filter(pl.col("session_date").is_between(expected_start, expected_end, closed="both"))
        invalid = (scoped.height != expected_sessions or scoped.get_column("session_date").null_count()
                   or scoped.get_column("adjusted_close").null_count()
                   or scoped.get_column("session_date").is_duplicated().any()
                   or scoped.get_column("session_date").to_list() != sorted(scoped.get_column("session_date").to_list())
                   or (scoped.get_column("adjusted_close") <= 0).any()
                   or scoped.get_column("session_date").min() != expected_start
                   or scoped.get_column("session_date").max() != expected_end)
        if invalid:
            raise RuntimeError(f"{symbol}: cobertura Gold inválida para la ventana congelada.")
        calendars.append(set(scoped.get_column("session_date").to_list()))
        COLAB_INPUT_AUDIT.append({"symbol": symbol, "sha256": expected_hash, "rows": scoped.height})

    common_dates = set.intersection(*calendars)
    if len(common_dates) != expected_sessions or min(common_dates) != expected_start or max(common_dates) != expected_end:
        raise RuntimeError("La intersección de calendarios Gold no coincide con las 2,232 sesiones esperadas.")
    REFERENCE_PATH = PROJECT_ROOT / "02_factor_momentum" / "notebooks" / "02_factor_momentum_signal_test.ipynb"
    REFERENCE_SHA256 = "7b6b96144cbf35ef19f8e44ad26d4706e1effc2c8669e185d53692ff2adf8abc"
    REFERENCE_URL = ("https://raw.githubusercontent.com/ADRIANVM117/quant-research-notes/"
                     f"{SOURCE_COMMIT}/Momentum_systematic_strategies/02_factor_momentum/notebooks/"
                     "02_factor_momentum_signal_test.ipynb")
    REFERENCE_PATH.parent.mkdir(parents=True, exist_ok=True)
    if REFERENCE_PATH.exists():
        if sha256(REFERENCE_PATH.read_bytes()).hexdigest() != REFERENCE_SHA256:
            raise RuntimeError("La notebook de referencia local no coincide con la revisión fijada; no se reemplaza.")
    else:
        try:
            with urlopen(REFERENCE_URL, timeout=60) as response:
                if response.status != 200:
                    raise RuntimeError(f"Notebook de referencia: HTTP {response.status} al descargar {REFERENCE_URL}")
                reference_payload = response.read()
        except (HTTPError, URLError, TimeoutError) as error:
            raise RuntimeError("No fue posible descargar la notebook de referencia fijada.") from error
        if sha256(reference_payload).hexdigest() != REFERENCE_SHA256:
            raise RuntimeError("La notebook de referencia remota no coincide con su SHA-256 fijado.")
        reference_temporary = REFERENCE_PATH.with_suffix(".ipynb.download")
        reference_temporary.write_bytes(reference_payload)
        reference_temporary.replace(REFERENCE_PATH)

    try:
        reference_cells = json.loads(REFERENCE_PATH.read_text(encoding="utf-8")).get("cells", [])
    except json.JSONDecodeError as error:
        raise RuntimeError("La notebook de referencia descargada no contiene JSON válido.") from error
    required_reference_cells = {"imports-and-data", "returns-and-oos-attribution", "signal-construction", "strategy-returns"}
    available_reference_cells = {cell.get("id") for cell in reference_cells}
    if not required_reference_cells.issubset(available_reference_cells):
        raise RuntimeError("La notebook de referencia no conserva las celdas analíticas requeridas para la identidad de señales.")

    os.chdir(PROJECT_ROOT)
    print(f"Colab listo | commit: {SOURCE_COMMIT} | sesiones comunes: {len(common_dates)}")
    print(f"Gold verificado: {GOLD_ROOT}")
    print(f"Versiones: numpy={np.__version__}, polars={pl.__version__}, plotly={plotly.__version__}")
else:
    print("Ejecución local detectada: no se instalaron dependencias ni se descargaron datos.")


In [ ]:
# Infraestructura P1: fija Signal Test a P0; no altera cálculos.
if IN_COLAB:
    P0_COMMIT = "af9a5f44412cc940fc655af92f74669fe341e84b"
    P0_SIGNAL_SHA256 = "5c43aa1b60930b4f16d6e60e13f818ca1a3ae23569ec64a4cf96adc1b0c97c5e"
    loaded_nbformat = sys.modules.get("nbformat")
    if loaded_nbformat is not None and getattr(loaded_nbformat, "__version__", None) != "5.10.4":
        raise RuntimeError("nbformat ya está cargado con una versión incompatible. Reinicia el entorno Colab limpio.")
    try:
        installed_nbformat = version("nbformat")
    except PackageNotFoundError:
        installed_nbformat = None
    if installed_nbformat != "5.10.4":
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "nbformat==5.10.4"])
    import nbformat
    if nbformat.__version__ != "5.10.4":
        raise RuntimeError("nbformat no coincide con la versión reproducible fijada.")
    p0_url = ("https://raw.githubusercontent.com/ADRIANVM117/quant-research-notes/"
              f"{P0_COMMIT}/Momentum_systematic_strategies/02_factor_momentum/notebooks/"
              "02_factor_momentum_signal_test.ipynb")
    try:
        with urlopen(p0_url, timeout=60) as response:
            if response.status != 200:
                raise RuntimeError(f"Signal Test P0: HTTP {response.status}.")
            p0_payload = response.read()
    except (HTTPError, URLError, TimeoutError) as error:
        raise RuntimeError("No fue posible descargar Signal Test desde P0.") from error
    if sha256(p0_payload).hexdigest() != P0_SIGNAL_SHA256:
        raise RuntimeError("Signal Test P0 no coincide con su SHA-256 fijado.")
    REFERENCE_PATH.write_bytes(p0_payload)
    p0_cells = json.loads(REFERENCE_PATH.read_text(encoding="utf-8")).get("cells", [])
    required_p0_cells = {"imports-and-data", "returns-and-oos-attribution", "signal-construction", "strategy-returns"}
    if not required_p0_cells.issubset({cell.get("id") for cell in p0_cells}):
        raise RuntimeError("Signal Test P0 no conserva las celdas analíticas requeridas.")
    print(f"Referencia P0 verificada: {P0_COMMIT} | SHA-256: {P0_SIGNAL_SHA256}")
else:
    print("Ejecución local: P1 no descarga dependencias remotas.")


# 01 — XLE Systematic Momentum: viabilidad de ejecución

> Esta es una evaluación exploratoria de ejecución para una candidata seleccionada en el mismo historial. No es una validación independiente de alpha ni una simulación operativa de IBKR.

## <b>Especificación congelada</b>

La señal reproduce exactamente XLE Systematic Momentum de la notebook de origen: formación de 126 sesiones, OLS rolling de 252 sesiones con intercepto y MKT, SIZE y GV. La posición es long/cash sin apalancamiento. Buy & Hold, TSMOM total y Market Momentum se mantienen como referencias.

In [ ]:
from datetime import date
from pathlib import Path
import json

import numpy as np
import plotly.graph_objects as go
import polars as pl
from IPython.display import HTML, display

SYMBOLS = ("SPY", "XLE", "IWM", "IWD", "IWF")
RESEARCH_START = date(2018, 1, 2)
RESEARCH_END = date(2026, 11, 17)
OLS_WINDOW = 252
MOMENTUM_HORIZON = 126
INITIAL_CAPITAL = 10_000.0
SLIPPAGE_BPS = 5.0

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "data" / "gold" / "daily_series" / "etf").is_dir():
            return candidate
        project = candidate / "Momentum_systematic_strategies"
        if (project / "data" / "gold" / "daily_series" / "etf").is_dir():
            return project
    raise FileNotFoundError("No se encontró data/gold/daily_series/etf.")

PROJECT_ROOT = find_project_root(Path.cwd())
GOLD_ROOT = PROJECT_ROOT / "data" / "gold" / "daily_series" / "etf"
frames = {}
for symbol in SYMBOLS:
    path = GOLD_ROOT / f"{symbol}.parquet"
    if not path.exists():
        raise FileNotFoundError(f"Gold no disponible para {symbol}: {path}")
    frames[symbol] = (
        pl.read_parquet(path)
        .with_columns(pl.col("session_date").cast(pl.Date))
        .filter(pl.col("session_date").is_between(RESEARCH_START, RESEARCH_END, closed="both"))
        .sort("session_date")
    )

xle_ohlc = frames["XLE"].select("session_date", "open", "close", "dividend_amount", "split_coefficient")
xle_ohlc.head()

## <b>Reproducción de la señal</b>

> Los precios ajustados se usan exclusivamente para retornos, atribución y señal. Los fills posteriores usan `open` sin ajustar. La verificación carga la notebook original en memoria y compara los pesos en fechas comunes.

In [ ]:
prices = frames["SPY"].select(pl.col("session_date").alias("date"), pl.col("adjusted_close").alias("SPY"))
for symbol in SYMBOLS[1:]:
    prices = prices.join(
        frames[symbol].select(pl.col("session_date").alias("date"), pl.col("adjusted_close").alias(symbol)),
        on="date",
        how="inner",
    )

returns = prices.sort("date").with_columns([pl.col(symbol).log().diff().alias(f"{symbol}_ret") for symbol in SYMBOLS]).drop_nulls()
model = returns.select("date", "XLE_ret", "SPY_ret", "IWM_ret", "IWF_ret", "IWD_ret").with_columns(
    pl.col("SPY_ret").alias("market"),
    (pl.col("IWM_ret") - pl.col("SPY_ret")).alias("size"),
    (pl.col("IWF_ret") - pl.col("IWD_ret")).alias("growth_value"),
)

dates = model.get_column("date").to_list()
factor_matrix = model.select("market", "size", "growth_value").to_numpy()
xle_returns = model.get_column("XLE_ret").to_numpy()
systematic_rows = []
for index in range(OLS_WINDOW, len(xle_returns)):
    design = np.column_stack((np.ones(OLS_WINDOW), factor_matrix[index - OLS_WINDOW:index]))
    coefficients = np.linalg.lstsq(design, xle_returns[index - OLS_WINDOW:index], rcond=None)[0]
    systematic_rows.append({
        "date": dates[index],
        "beta_asof_date": dates[index - 1],
        "actual_return": xle_returns[index],
        "market_return": factor_matrix[index, 0],
        "systematic_return": coefficients[0] + factor_matrix[index] @ coefficients[1:],
    })

attribution = pl.DataFrame(systematic_rows).with_columns((pl.col("actual_return") - pl.col("systematic_return")).alias("residual_return"))
signal_rows = []
for end in range(MOMENTUM_HORIZON - 1, attribution.height):
    window = attribution.slice(end - MOMENTUM_HORIZON + 1, MOMENTUM_HORIZON)
    signal_rows.append({
        "signal_date": window.get_column("date")[-1],
        "tsmom_weight": float(window.get_column("actual_return").sum() > 0),
        "market_weight": float(window.get_column("market_return").sum() > 0),
        "systematic_weight": float(window.get_column("systematic_return").sum() > 0),
    })

signals = pl.DataFrame(signal_rows)
return_dates = model.select(pl.col("date").alias("signal_date"), pl.col("date").shift(-1).alias("return_date"))
execution_schedule = signals.join(return_dates, on="signal_date", how="inner").join(
    xle_ohlc.rename({"session_date": "return_date"}), on="return_date", how="inner"
).join(
    xle_ohlc.select(pl.col("session_date").alias("signal_date"), pl.col("close").alias("signal_close")),
    on="signal_date", how="inner"
).drop_nulls().sort("signal_date")
execution_schedule.head()

In [ ]:
source_notebook = PROJECT_ROOT / "02_factor_momentum" / "notebooks" / "02_factor_momentum_signal_test.ipynb"
reference_notebook = json.loads(source_notebook.read_text(encoding="utf-8"))
reference_namespace = {"__name__": "__reference__"}
OMITTED_REFERENCE_INFRASTRUCTURE = {"colab-bootstrap-signal"}
for cell in reference_notebook["cells"]:
    if cell["cell_type"] == "code" and cell.get("id") not in OMITTED_REFERENCE_INFRASTRUCTURE:
        exec(compile("".join(cell["source"]), str(source_notebook), "exec"), reference_namespace)
    if cell.get("id") == "strategy-returns":
        break

reference_weights = reference_namespace["strategy_returns"].filter((pl.col("asset") == "XLE") & pl.col("alternative").is_in(["TSMOM total", "Market momentum", "Systematic momentum"])).select("signal_date", "return_date", "alternative", "weight")
candidate_weights = pl.concat([
    execution_schedule.select("signal_date", "return_date", pl.lit("TSMOM total").alias("alternative"), pl.col("tsmom_weight").alias("weight")),
    execution_schedule.select("signal_date", "return_date", pl.lit("Market momentum").alias("alternative"), pl.col("market_weight").alias("weight")),
    execution_schedule.select("signal_date", "return_date", pl.lit("Systematic momentum").alias("alternative"), pl.col("systematic_weight").alias("weight")),
])
signal_reproduction = candidate_weights.join(reference_weights, on=["signal_date", "return_date", "alternative"], how="inner", suffix="_reference")
signal_reproduction_audit = pl.DataFrame([{
    "common_rows": signal_reproduction.height,
    "max_abs_weight_difference": (signal_reproduction["weight"] - signal_reproduction["weight_reference"]).abs().max(),
}])
if signal_reproduction.height != reference_weights.height or signal_reproduction_audit["max_abs_weight_difference"][0] != 0:
    raise AssertionError("Las señales no reproducen la notebook de origen.")
signal_reproduction_audit

## <b>Supuestos de órdenes y contabilidad</b>

> Se reutilizan selectivamente del motor de rebalancing las ideas de orden preparada al cierre, fill adverso, compra financiable, cash/shares y ledger. No se reutiliza volatility targeting ni su lógica de rebalanceo.

Cada compra se dimensiona con el cierre sin ajustar conocido en `signal_date`; en la apertura siguiente se reduce si el fill real no es financiable. Los dividendos se acreditan como efectivo y los splits ajustan las acciones en la fecha registrada por Gold. Esto es una aproximación contable basada en la convención del proveedor.

In [ ]:
SCENARIOS = {
    "Open, no costs": {"commission": False, "slippage_bps": 0.0},
    "IBKR Pro Fixed": {"commission": True, "slippage_bps": 0.0},
    "IBKR Pro Fixed + 5 bps": {"commission": True, "slippage_bps": SLIPPAGE_BPS},
}
STRATEGIES = {
    "Buy & Hold": None,
    "TSMOM total": "tsmom_weight",
    "Market momentum": "market_weight",
    "Systematic momentum": "systematic_weight",
}

def ibkr_fixed_commission(shares, fill_price, enabled):
    if not enabled or shares == 0:
        return 0.0
    trade_value = abs(shares) * fill_price
    return min(max(abs(shares) * 0.005, 1.0), trade_value * 0.01)

def affordable_shares(cash, estimated_price, commission_enabled):
    quantity = int(cash // estimated_price)
    while quantity and quantity * estimated_price + ibkr_fixed_commission(quantity, estimated_price, commission_enabled) > cash:
        quantity -= 1
    return quantity

def run_execution(schedule, strategy, scenario):
    settings = SCENARIOS[scenario]
    shares, cash = 0, INITIAL_CAPITAL
    ledger, operations = [], []
    for row in schedule.iter_rows(named=True):
        split = row["split_coefficient"] or 1.0
        if split != 1.0:
            shares = int(round(shares * split))
        dividend_cash = shares * (row["dividend_amount"] or 0.0)
        cash += dividend_cash
        desired = 1.0 if strategy == "Buy & Hold" else row[STRATEGIES[strategy]]
        trade_shares, fill_price, commission, slippage_cost = 0, row["open"], 0.0, 0.0
        if desired == 0 and shares:
            trade_shares = -shares
        elif desired == 1 and shares == 0:
            estimated_fill = row["signal_close"] * (1 + settings["slippage_bps"] / 10_000)
            planned = affordable_shares(cash, estimated_fill, settings["commission"])
            actual_fill = row["open"] * (1 + settings["slippage_bps"] / 10_000)
            trade_shares = min(planned, affordable_shares(cash, actual_fill, settings["commission"]))
        if trade_shares:
            fill_price = row["open"] * (1 + np.sign(trade_shares) * settings["slippage_bps"] / 10_000)
            commission = ibkr_fixed_commission(trade_shares, fill_price, settings["commission"])
            slippage_cost = abs(trade_shares) * abs(fill_price - row["open"])
            if trade_shares > 0:
                cash -= trade_shares * fill_price + commission
            else:
                cash += abs(trade_shares) * fill_price - commission
            shares += trade_shares
            operations.append({
                "date": row["return_date"], "strategy": strategy, "scenario": scenario,
                "shares": trade_shares, "fill_price": fill_price, "open_price": row["open"],
                "notional": abs(trade_shares) * fill_price, "commission": commission, "slippage_cost": slippage_cost,
            })
        if cash < -1e-8:
            raise AssertionError("La simulación produjo efectivo negativo.")
        ledger.append({
            "date": row["return_date"], "signal_date": row["signal_date"], "strategy": strategy, "scenario": scenario,
            "desired_weight": desired, "shares": shares, "cash": cash, "close": row["close"],
            "nav": cash + shares * row["close"], "dividend_cash": dividend_cash,
        })
    return pl.DataFrame(ledger), pl.DataFrame(operations) if operations else pl.DataFrame(schema={"date": pl.Date, "strategy": pl.String, "scenario": pl.String, "shares": pl.Int64, "fill_price": pl.Float64, "open_price": pl.Float64, "notional": pl.Float64, "commission": pl.Float64, "slippage_cost": pl.Float64})


In [ ]:
ledger_frames, operation_frames = [], []
for scenario in SCENARIOS:
    for strategy in STRATEGIES:
        ledger, operations = run_execution(execution_schedule, strategy, scenario)
        ledger_frames.append(ledger)
        operation_frames.append(operations)

ledger = pl.concat(ledger_frames)
operations = pl.concat(operation_frames)
ledger.select("date", "strategy", "scenario", "shares", "cash", "nav").tail()

## <b>Escenarios de fricción</b>

La comisión usa IBKR Pro Fixed para acciones/ETFs estadounidenses: USD 0.005 por acción, mínimo USD 1 y tope 1% del valor negociado, consultado el 2026-11-19. El escenario de 5 bps es slippage adverso ilustrativo, no bid/ask histórico ni coste real medido de IBKR.

In [ ]:
summary_rows = []
for scenario in SCENARIOS:
    for strategy in STRATEGIES:
        history = ledger.filter((pl.col("scenario") == scenario) & (pl.col("strategy") == strategy)).sort("date")
        nav = history.get_column("nav").to_numpy()
        nav_points = np.r_[INITIAL_CAPITAL, nav]
        log_returns = np.diff(np.log(nav_points))
        drawdown = nav_points / np.maximum.accumulate(nav_points) - 1
        trades = operations.filter((pl.col("scenario") == scenario) & (pl.col("strategy") == strategy))
        summary_rows.append({
            "scenario": scenario, "strategy": strategy, "final_capital": nav[-1],
            "cagr_pct": ((nav[-1] / INITIAL_CAPITAL) ** (252 / len(log_returns)) - 1) * 100,
            "annualized_volatility_pct": log_returns.std(ddof=1) * np.sqrt(252) * 100,
            "sharpe": log_returns.mean() / log_returns.std(ddof=1) * np.sqrt(252),
            "max_drawdown_pct": drawdown.min() * 100, "operations": trades.height,
            "turnover_multiple": trades.get_column("notional").sum() / INITIAL_CAPITAL if trades.height else 0.0,
            "commission_total": trades.get_column("commission").sum() if trades.height else 0.0,
            "slippage_total": trades.get_column("slippage_cost").sum() if trades.height else 0.0,
            "time_invested_pct": (history.get_column("shares") > 0).mean() * 100,
        })

execution_summary = pl.DataFrame(summary_rows).with_columns(
    pl.all().exclude("scenario", "strategy", "operations").round(3)
)
execution_summary

In [ ]:
ideal_rows = []
for strategy in STRATEGIES:
    sample = reference_namespace["strategy_returns"].filter((pl.col("asset") == "XLE") & (pl.col("alternative") == strategy)).sort("return_date")
    returns = sample.get_column("strategy_return").to_numpy()
    wealth = np.exp(np.cumsum(returns))
    ideal_rows.append({
        "strategy": strategy,
        "idealized_return_pct": (np.exp(returns.mean() * 252) - 1) * 100,
        "idealized_sharpe": returns.mean() / returns.std(ddof=1) * np.sqrt(252),
        "idealized_max_drawdown_pct": (wealth / np.maximum.accumulate(wealth) - 1).min() * 100,
    })
idealized = pl.DataFrame(ideal_rows)
open_execution = execution_summary.filter(pl.col("scenario") == "Open, no costs").select(
    "strategy", pl.col("cagr_pct").alias("open_execution_cagr_pct"),
    pl.col("sharpe").alias("open_execution_sharpe"),
    pl.col("max_drawdown_pct").alias("open_execution_max_drawdown_pct"),
)
idealized_vs_execution = idealized.join(open_execution, on="strategy").with_columns(pl.all().exclude("strategy").round(3))
idealized_vs_execution

In [ ]:
figure = go.Figure()
for strategy in STRATEGIES:
    history = ledger.filter((pl.col("scenario") == "IBKR Pro Fixed + 5 bps") & (pl.col("strategy") == strategy)).sort("date")
    figure.add_scatter(x=history["date"], y=history["nav"], mode="lines", name=strategy)

figure.update_layout(title="XLE — capital tras comisiones y slippage ilustrativo", xaxis_title="fecha", yaxis_title="USD", template="plotly_white")
display(HTML(figure.to_html(include_plotlyjs="cdn", full_html=False)))

In [ ]:
systematic = ledger.filter((pl.col("scenario") == "IBKR Pro Fixed + 5 bps") & (pl.col("strategy") == "Systematic momentum")).sort("date")
tsmom = ledger.filter((pl.col("scenario") == "IBKR Pro Fixed + 5 bps") & (pl.col("strategy") == "TSMOM total")).sort("date")
cumulative_difference = (np.log(systematic["nav"].to_numpy() / INITIAL_CAPITAL) - np.log(tsmom["nav"].to_numpy() / INITIAL_CAPITAL)) * 100

figure = go.Figure()
figure.add_scatter(x=systematic["date"], y=cumulative_difference, mode="lines", name="Systematic − TSMOM")
figure.add_hline(y=0, line_dash="dash", line_color="gray")
figure.update_layout(title="XLE — diferencial acumulado de ejecución", xaxis_title="fecha", yaxis_title="diferencia de capital logarítmico (%)", template="plotly_white")
display(HTML(figure.to_html(include_plotlyjs="cdn", full_html=False)))

## <b>Interpretación</b>

- Open, no costs deja Systematic Momentum prácticamente alineado con la versión idealizada (15.623% frente a 15.571% CAGR); la diferencia procede de aperturas, acciones enteras, cash y la contabilidad explícita de dividendos.
- Con comisión IBKR Pro Fixed y 5 bps ilustrativos, Systematic Momentum cae de 15.623% a 15.046% CAGR y de USD 29,078 a USD 28,028 de capital final.
- Bajo esas fricciones, la señal sistemática conserva ventaja frente a TSMOM (3.929% CAGR) y Market Momentum (11.824%), con menor drawdown que ambos.
- Systematic Momentum registra 63 operaciones, cerca de 8.6 por año, una frecuencia manejable para un retail pero todavía sensible a fills y costes no observados.
- La contabilidad procesa 30 fechas de dividendos en la ventana efectiva de ejecución y un split desde Gold. Sus fechas son una aproximación del proveedor, no un calendario de pago o confirmación de broker.

## <b>Conclusión</b>

- La ejecución con aperturas, acciones enteras y cash no negativo reduce la rentabilidad sistemática sólo moderadamente bajo los supuestos usados; no elimina su ventaja histórica en XLE.
- La señal conserva ventaja frente a TSMOM y Market Momentum tras costes, mientras que Buy & Hold termina con menor capital y un drawdown mucho mayor.
- Las 63 operaciones históricas son operativamente razonables para un retail, pero los USD 68 de comisión y USD 456 de slippage ilustrativo no representan costes medidos de una cuenta IBKR.
- Bid/ask, profundidad, fills parciales, impuestos, dividendos pagados en fecha distinta, fees regulatorios y el sesgo de selección de XLE impiden tratar el resultado como PnL real o alpha confirmado.
- La candidata justifica una observación prospectiva en paper trading sólo con esta especificación congelada y con nuevas observaciones fuera del historial de selección.

Una observación prospectiva en paper trading sólo tendría sentido con esta señal congelada y con controles explícitos de órdenes, corporate actions y costes reales.